In [1]:
import os
import time
from pathlib import Path
import pandas as pd
import numpy as np

# Reproducibility
SEED = 42
np.random.seed(SEED)

# Paths
PROJECT_ROOT = Path.home() / "icidea_llm_ids"
DATA_DIR = PROJECT_ROOT / "data"
ARTIFACTS_DIR = PROJECT_ROOT / "artifacts"
ARTIFACTS_DIR.mkdir(exist_ok=True)

# Per-class sampling budget
# Rationale: we need enough rows to build ~5k-10k windows of 29 frames each per class
# 200k rows per class = up to ~6900 windows = comfortably enough
SAMPLES_PER_CLASS = 200_000

# File registry
FILES = {
    "NORMAL": DATA_DIR / "normal_run_data.txt",
    "DOS":    DATA_DIR / "DoS_dataset.csv",
    "FUZZY":  DATA_DIR / "Fuzzy_dataset.csv",
    "GEAR":   DATA_DIR / "gear_dataset.csv",
    "RPM":    DATA_DIR / "RPM_dataset.csv",
}

LABEL_MAP = {
    "NORMAL": 0,
    "DOS":    1,
    "FUZZY":  2,
    "GEAR":   3,
    "RPM":    4,
}

# Sanity check: verify all files exist
for name, path in FILES.items():
    assert path.exists(), f"Missing file: {path}"
    size_mb = path.stat().st_size / 1e6
    print(f"✓ {name:7s}  {path.name:30s}  {size_mb:7.1f} MB")

✓ NORMAL   normal_run_data.txt                87.4 MB
✓ DOS      DoS_dataset.csv                   190.1 MB
✓ FUZZY    Fuzzy_dataset.csv                 198.5 MB
✓ GEAR     gear_dataset.csv                  230.3 MB
✓ RPM      RPM_dataset.csv                   239.6 MB


In [3]:
def reformat_hcrl_row(row):
    """
    Handles HCRL's variable-DLC quirk.

    In HCRL CSV files, when DLC < 8:
      - DATA0..DATA(DLC-1) contain valid payload bytes (hex strings)
      - DATA(DLC) contains the label letter ('R' or 'T'), NOT a payload byte
      - DATA(DLC+1)..DATA7 are empty/NaN

    When DLC == 8:
      - DATA0..DATA7 contain valid payload bytes
      - The label letter is in the 'label' column

    This function fixes that, producing rows where:
      - DATA0..DATA7 are always integers (0-255)
      - 'label' is always the letter ('R' or 'T')

    Inherited from CAN-IDSs-on-the-ROAD (Guerra et al. 2024, util/data.py).
    """
    dlc = int(row["DLC"])
    if dlc != 8:
        # Move misplaced label letter from data column to label column
        row["label"] = row[f"DATA{dlc}"]
        # Convert valid payload bytes from hex to int
        for i in range(dlc):
            row[f"DATA{i}"] = int(row[f"DATA{i}"], 16)
        # Zero-pad missing payload bytes
        for i in range(dlc, 8):
            row[f"DATA{i}"] = 0
    else:
        # All 8 bytes are payload
        for i in range(8):
            row[f"DATA{i}"] = int(row[f"DATA{i}"], 16)
    return row

In [4]:
def load_attack_file(path, class_label_int, n_samples, seed=42):
    """
    Load an HCRL attack CSV.

    Files have NO header. Column order:
        timestamp, ID, DLC, DATA0..DATA7, label

    The 'label' column contains 'R' (normal/regular) or 'T' (attack/target).

    We do NOT collapse all attack rows to a single class label here -- the
    file ALREADY indicates which rows are the injected attack class. So:
      - Rows with label='T' get class_label_int (DOS=1, FUZZY=2, etc.)
      - Rows with label='R' are benign frames mixed in -- we discard them
        from attack files (we get our NORMAL data from normal_run_data.txt)
    """
    print(f"Loading {path.name}...")
    t0 = time.time()

    cols = ["timestamp", "ID", "DLC",
            "DATA0", "DATA1", "DATA2", "DATA3",
            "DATA4", "DATA5", "DATA6", "DATA7",
            "label"]
    df = pd.read_csv(path, names=cols, header=None, low_memory=False)
    print(f"  Raw rows: {len(df):,}")
    print(f"  Label distribution: {df['label'].value_counts().to_dict()}")

    # Apply HCRL DLC fix (slow but necessary; ~1-2 min per file)
    print(f"  Applying HCRL row reformatter (may take ~30-90s)...")
    df = df.apply(reformat_hcrl_row, axis=1)

    # Convert ID from hex string to int
    df["ID"] = df["ID"].apply(lambda x: int(x, 16))

    # Keep only attack rows (label='T')
    attack_rows = df[df["label"] == "T"].copy()
    print(f"  Attack rows: {len(attack_rows):,}")

    # Sample down
    n = min(n_samples, len(attack_rows))
    sampled = attack_rows.sample(n=n, random_state=seed).reset_index(drop=True)

    # Assign class label
    sampled["class"] = class_label_int

    # Drop the original 'label' column (R/T letter) — we don't need it anymore
    sampled = sampled.drop(columns=["label"])

    print(f"  Sampled {n:,} attack rows in {time.time()-t0:.1f}s")
    return sampled

In [5]:
def load_normal_file(path, class_label_int, n_samples, seed=42):
    """
    Load HCRL normal_run_data.txt — attack-free baseline traffic.

    Format is DIFFERENT from the attack CSVs. It's a space-separated log:
        Timestamp: 1478195722.310891        ID: 0153    000    DLC: 8    00 00 00 00 00 00 00 00

    We parse it line by line into the same schema as the attack DataFrames.
    """
    print(f"Loading {path.name}...")
    t0 = time.time()

    rows = []
    with open(path, "r") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            # Format: "Timestamp:      1478195722.310891   ID: 0153    000    DLC: 8    fd c4 14 67 ff 20 00 25"
            try:
                # Split on whitespace
                parts = line.split()
                # Find indices of known tokens
                ts = float(parts[1])
                can_id_str = parts[3]
                dlc = int(parts[6])
                payload_hex = parts[7:7+dlc] if dlc > 0 else []
                # Pad payload to 8 bytes
                payload = [int(b, 16) for b in payload_hex] + [0] * (8 - dlc)

                rows.append({
                    "timestamp": ts,
                    "ID": int(can_id_str, 16),
                    "DLC": dlc,
                    **{f"DATA{i}": payload[i] for i in range(8)},
                })
            except (ValueError, IndexError) as e:
                # Skip malformed lines (rare in HCRL)
                continue

    df = pd.DataFrame(rows)
    print(f"  Raw rows: {len(df):,}")

    # Sample down
    n = min(n_samples, len(df))
    sampled = df.sample(n=n, random_state=seed).reset_index(drop=True)
    sampled["class"] = class_label_int

    print(f"  Sampled {n:,} normal rows in {time.time()-t0:.1f}s")
    return sampled

In [6]:
# Load all 5 files
dfs = {}

dfs["NORMAL"] = load_normal_file(FILES["NORMAL"], LABEL_MAP["NORMAL"], SAMPLES_PER_CLASS, seed=SEED)
print()
dfs["DOS"]    = load_attack_file(FILES["DOS"],    LABEL_MAP["DOS"],    SAMPLES_PER_CLASS, seed=SEED)
print()
dfs["FUZZY"]  = load_attack_file(FILES["FUZZY"],  LABEL_MAP["FUZZY"],  SAMPLES_PER_CLASS, seed=SEED)
print()
dfs["GEAR"]   = load_attack_file(FILES["GEAR"],   LABEL_MAP["GEAR"],   SAMPLES_PER_CLASS, seed=SEED)
print()
dfs["RPM"]    = load_attack_file(FILES["RPM"],    LABEL_MAP["RPM"],    SAMPLES_PER_CLASS, seed=SEED)

Loading normal_run_data.txt...
  Raw rows: 988,871
  Sampled 200,000 normal rows in 4.3s

Loading DoS_dataset.csv...
  Raw rows: 3,665,771
  Label distribution: {'R': 3047062, 'T': 587521}
  Applying HCRL row reformatter (may take ~30-90s)...
  Attack rows: 587,521
  Sampled 200,000 attack rows in 275.5s

Loading Fuzzy_dataset.csv...
  Raw rows: 3,838,860
  Label distribution: {'R': 3259177, 'T': 491847}
  Applying HCRL row reformatter (may take ~30-90s)...
  Attack rows: 491,847
  Sampled 200,000 attack rows in 273.1s

Loading gear_dataset.csv...
  Raw rows: 4,443,142
  Label distribution: {'R': 3805725, 'T': 597252}
  Applying HCRL row reformatter (may take ~30-90s)...
  Attack rows: 597,252
  Sampled 200,000 attack rows in 330.1s

Loading RPM_dataset.csv...
  Raw rows: 4,621,702
  Label distribution: {'R': 3925329, 'T': 654897}
  Applying HCRL row reformatter (may take ~30-90s)...
  Attack rows: 654,897
  Sampled 200,000 attack rows in 314.6s


In [7]:
# Combine into single DataFrame
full_df = pd.concat(dfs.values(), ignore_index=True)

print("="*60)
print("COMBINED DATASET SUMMARY")
print("="*60)
print(f"Total rows: {len(full_df):,}")
print(f"\nClass distribution:")
print(full_df["class"].value_counts().sort_index().to_string())
print(f"\nClass labels:")
for name, idx in LABEL_MAP.items():
    print(f"  {idx} = {name}")
print(f"\nColumn dtypes:")
print(full_df.dtypes.to_string())
print(f"\nFirst 5 rows:")
print(full_df.head())
print(f"\nMemory usage: {full_df.memory_usage(deep=True).sum() / 1e6:.1f} MB")

COMBINED DATASET SUMMARY
Total rows: 1,000,000

Class distribution:
class
0    200000
1    200000
2    200000
3    200000
4    200000

Class labels:
  0 = NORMAL
  1 = DOS
  2 = FUZZY
  3 = GEAR
  4 = RPM

Column dtypes:
timestamp    float64
ID             int64
DLC            int64
DATA0          int64
DATA1          int64
DATA2          int64
DATA3          int64
DATA4          int64
DATA5          int64
DATA6          int64
DATA7          int64
class          int64

First 5 rows:
      timestamp    ID  DLC  DATA0  DATA1  DATA2  DATA3  DATA4  DATA5  DATA6  \
0  1.479122e+09   809    8    220    184    127     20     17     32      0   
1  1.479122e+09   497    8      0    158     99     32      6     98     16   
2  1.479122e+09  1349    8    216      0      0    142      0      0      0   
3  1.479122e+09   790    8      5     44    252     17     44     27     37   
4  1.479122e+09   608    8     71    152    161     48    255    175    120   

   DATA7  class  
0     20      0  
1

In [8]:
# Save to parquet for fast reload in later sections
artifact_path = ARTIFACTS_DIR / "section3_loaded_data.parquet"
full_df.to_parquet(artifact_path, index=False)
size_mb = artifact_path.stat().st_size / 1e6
print(f"✓ Saved to {artifact_path}")
print(f"  Size: {size_mb:.1f} MB")
print(f"  Sections 4+ will load from this file (fast, ~1s).")

✓ Saved to /Users/deepakpatnaik/icidea_llm_ids/artifacts/section3_loaded_data.parquet
  Size: 10.3 MB
  Sections 4+ will load from this file (fast, ~1s).


In [9]:
print("="*60)
print("COMBINED DATASET SUMMARY")
print("="*60)
print(f"Total rows: {len(full_df):,}")
print(f"\nClass distribution:")
print(full_df["class"].value_counts().sort_index().to_string())
print(f"\nColumn dtypes:")
print(full_df.dtypes.to_string())
print(f"\nFirst 5 rows:")
print(full_df.head())
print(f"\nMemory usage: {full_df.memory_usage(deep=True).sum() / 1e6:.1f} MB")

COMBINED DATASET SUMMARY
Total rows: 1,000,000

Class distribution:
class
0    200000
1    200000
2    200000
3    200000
4    200000

Column dtypes:
timestamp    float64
ID             int64
DLC            int64
DATA0          int64
DATA1          int64
DATA2          int64
DATA3          int64
DATA4          int64
DATA5          int64
DATA6          int64
DATA7          int64
class          int64

First 5 rows:
      timestamp    ID  DLC  DATA0  DATA1  DATA2  DATA3  DATA4  DATA5  DATA6  \
0  1.479122e+09   809    8    220    184    127     20     17     32      0   
1  1.479122e+09   497    8      0    158     99     32      6     98     16   
2  1.479122e+09  1349    8    216      0      0    142      0      0      0   
3  1.479122e+09   790    8      5     44    252     17     44     27     37   
4  1.479122e+09   608    8     71    152    161     48    255    175    120   

   DATA7  class  
0     20      0  
1      6      0  
2      0      0  
3    123      0  
4     72      0  
